# Stage 5 Structured Output Evaluation

Оценка JSON/тегов/описаний для Qwen и hybrid outputs. Скопируйте `drop_in/scripts/evaluate_structured_json_output.py` в `scripts/` репозитория.

In [ ]:

from pathlib import Path
import subprocess, json, os, shutil, csv
import pandas as pd
from IPython.display import display, Markdown

REPO_URL = os.environ.get("REPO_URL", "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git")
REPO_DIR = Path(os.environ.get("REPO_DIR", "/kaggle/working/vlm-for-insulator-defect-detection"))
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
    Path("/kaggle/input"),
]
TRAIN_REL = Path("stage3_regrouped_v2/train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl")
VAL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")

def sh(args, cwd=None, check=True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed {p.returncode}: {' '.join(str(a) for a in args)}")
    return p

def find_data_root():
    for root in DATASET_ROOT_CANDIDATES:
        if (root/TRAIN_REL).exists() and (root/VAL_REL).exists():
            return root
    for root in Path('/kaggle/input').glob('**') if Path('/kaggle/input').exists() else []:
        if (root/TRAIN_REL).exists() and (root/VAL_REL).exists():
            return root
    raise FileNotFoundError('Could not find stage3_regrouped_v2 train/val JSONL')


In [ ]:
DATA_ROOT = find_data_root()
GT_JSONL = DATA_ROOT / VAL_REL
print('GT_JSONL:', GT_JSONL)
if not REPO_DIR.exists():
    sh(['git','clone',REPO_URL,str(REPO_DIR)])
sh(['python','-m','pip','install','-q','pandas'], cwd=REPO_DIR)


## Найти prediction JSONL

In [ ]:
# Candidate paths. Update manually if the files were downloaded into another folder.
candidates = [
    REPO_DIR/'outputs/stage3_vlm_baseline_runs/stage3_qwen_val_v2_clean_final/predictions_vlm_labels_v1.jsonl',
    REPO_DIR/'reports/operation_next/stage4_context_pad030_qwen_artifacts/qwen_predictions_vlm_labels_v1.jsonl',
]
for p in candidates:
    print(p, p.exists())

# Fallback: list possible files.
found = list(REPO_DIR.glob('**/*predictions_vlm_labels_v1*.jsonl'))[:50]
print('Found prediction files:', len(found))
for p in found[:20]:
    print(p)


## Запуск для выбранного файла

In [ ]:
# Set this to the prediction file you want to evaluate.
PRED_JSONL = candidates[0] if candidates[0].exists() else (found[0] if found else None)
if PRED_JSONL is None:
    raise FileNotFoundError('No predictions_vlm_labels_v1 JSONL found. Download run artifacts first.')
RUN_NAME = PRED_JSONL.parent.name
print('Evaluating:', PRED_JSONL)
sh([
    'python','scripts/evaluate_structured_json_output.py',
    '--gt-jsonl', GT_JSONL,
    '--pred-jsonl', PRED_JSONL,
    '--run-name', RUN_NAME,
    '--out-dir', 'reports/next_research/structured_output_eval',
], cwd=REPO_DIR)


In [ ]:
summary_path = REPO_DIR/'reports/next_research/structured_output_eval'/RUN_NAME/'summary.md'
print(summary_path.read_text(encoding='utf-8'))
review_path = REPO_DIR/'reports/next_research/structured_output_eval'/RUN_NAME/'manual_review_template.csv'
review = pd.read_csv(review_path)
display(review.head(20))


## После ручной оценки

In [ ]:
# After filling manual_review_template.csv, rerun with --manual-review-csv.
# sh([... '--manual-review-csv', review_path ...])
